# Parsing Miniprot results

In [124]:
import pandas as pd
import matplotlib
import numpy as np
import matplotlib.pyplot as plt

In [125]:
# barn owl microchromosome names
barn_owlMO_microchromosomes = ['Mo_LG41', 'Mo_LG42', 'Mo_LG43', 'Mo_LG44', 'Mo_LG45', 'Mo_LG46']
barn_owl_microchromosomes = ['Fa_LG41', 'Fa_LG42', 'Fa_LG43', 'Fa_LG44', 'Fa_LG45', 'Fa_LG46']

In [ ]:
# paf parsing
def format_paf(paf):
    paf.columns = ['qname', 'qlen', 'qstart', 'qend', 'strand', 'tname', 'tlen', 'tstart', 'tend', 'nmatch', 'alen', 'mapq', 'AS', 'ms', 'np', 'fs', 'st', 'da', 'do', 'cg', 'cs']
    paf['AS'] = paf['AS'].str.replace('AS:i:', '').astype(int)
    paf['ms'] = paf['ms'].str.replace('ms:i:', '').astype(int)
    paf['np'] = paf['np'].str.replace('np:i:', '').astype(int)
    paf['fs'] = paf['fs'].str.replace('fs:i:', '').astype(int)
    paf['st'] = paf['st'].str.replace('st:i:', '').astype(int)
    paf['da'] = paf['da'].str.replace('da:i:', '').astype(int)
    paf['do'] = paf['do'].str.replace('do:i:', '').astype(int)
    paf['p_cov'] = ((paf['qend'] - paf['qstart'])/paf['qlen']) * 100
    paf['p_id'] = (paf['nmatch']/paf['alen']) * 100
    return paf

## Chicken vs mother assembly

In [ ]:
# load chicken owl annotation 
chicken_data = pd.read_csv('refdata/GCF_016699485.2_bGalGal1.mat.broiler.GRCg7b_feature_table.txt', sep='\t', header=0)

# load chicken owl to barnMO owl PAF & format the columns
chicken_barnMO = pd.read_csv('pafs/barnowlMO_chicken.paf', sep='\t', header=None)
chicken_barnMO_paf = format_paf(chicken_barnMO)

C:\Users\mbachma3\AppData\Local\Temp\ipykernel_31900\3878072708.py:2: DtypeWarning: Columns (5,16) have mixed types. Specify dtype option on import or set low_memory=False.
  chicken_data = pd.read_csv('C:/Users/mbachma3/Documents/REMOTE_WORKING/refgenome2025/GCF_016699485.2_bGalGal1.mat.broiler.GRCg7b_feature_table.txt', sep='\t', header=0)


In [186]:
chicken_data_cds = chicken_data[chicken_data['# feature'] == 'CDS']

In [ ]:
# percent identity 
chicken_barnMO_paf['p_id'] = (chicken_barnMO_paf['nmatch']/chicken_barnMO_paf['alen']) * 100
# % of reference protein covered
chicken_barnMO_paf['p_cov'] = ((chicken_barnMO_paf['qend'] - chicken_barnMO_paf['qstart'])/chicken_barnMO_paf['qlen']) * 100

In [ ]:
chicken_barnMO_paf_filt = chicken_barnMO_paf[(chicken_barnMO_paf['p_cov'] > 90) & (chicken_barnMO_paf['p_id'] > 70)]
print('raw alignments:', chicken_barnMO_paf.shape[0])
print('alignments after filtering:', chicken_barnMO_paf_filt.shape[0])

raw alignments: 71105
alignments after filtering: 62041


### Merge paf data with feature information

In [ ]:
# change column names to avoid duplicates & allow merging
chicken_barnMO_paf_filt['product_accession'] = chicken_barnMO_paf_filt['qname']

chicken_data_cds.columns = ['feature', 'class', 'assembly', 'assembly_unit', 'seq_type',
       'chromosome', 'genomic_accession', 'feature_start', 'feature_end', 'feature_strand',
       'product_accession', 'non-redundant_refseq', 'related_accession',
       'name', 'symbol', 'GeneID', 'locus_tag', 'feature_interval_length',
       'product_length', 'attributes']

In [440]:
full_chicken_barnMO_paf = chicken_barnMO_paf_filt.merge(chicken_data_cds, on='product_accession', how='inner')

For Sankey diagrams, we are interested in the coordinates.

In [ ]:
sankey_chickenMO = full_chicken_barnMO_paf[['qname', 'chromosome', 'genomic_accession', 'tname']]

sankey_chickenMO['chr'] = np.where(
    sankey_chickenMO['chromosome'].isna(),
    'unplaced',
    sankey_chickenMO['chromosome']
)

sankey_chickenMO['chromosome'] = np.where(
    sankey_chickenMO['chromosome'].isna(),
    'unplaced_' + sankey_chickenMO['genomic_accession'].astype(str),
    sankey_chickenMO['chromosome']
)

In [ ]:
sankey_chickenMO.to_csv('pafs/sankey_chicken_to_barnowlMO.tsv', sep='\t', index=False)

## Chicken vs father assembly

In [ ]:
# load chicken owl to barn owl father PAF & format the columns
chicken_barn = pd.read_csv('pafs/barnowl_chicken.paf', sep='\t', header=None)
chicken_barn_paf = format_paf(chicken_barn)

In [ ]:
# percent identity 
chicken_barn_paf['p_id'] = (chicken_barn_paf['nmatch']/chicken_barn_paf['alen']) * 100
# % of reference protein covered
chicken_barn_paf['p_cov'] = ((chicken_barn_paf['qend'] - chicken_barn_paf['qstart'])/chicken_barn_paf['qlen']) * 100

In [217]:
chicken_barn_paf_filt = chicken_barn_paf[(chicken_barn_paf['p_cov'] > 90) & (chicken_barn_paf['p_id'] > 70)]
print('raw alignments:', chicken_barn_paf.shape[0])
print('alignments after filtering:', chicken_barn_paf_filt.shape[0])

raw alignments: 71036
alignments after filtering: 61810


### Merge paf data with feature information

In [ ]:
# change column names to avoid duplicates & allow merging
chicken_barn_paf_filt['product_accession'] = chicken_barn_paf_filt['qname']

chicken_data_cds.columns = ['feature', 'class', 'assembly', 'assembly_unit', 'seq_type',
       'chromosome', 'genomic_accession', 'feature_start', 'feature_end', 'feature_strand',
       'product_accession', 'non-redundant_refseq', 'related_accession',
       'name', 'symbol', 'GeneID', 'locus_tag', 'feature_interval_length',
       'product_length', 'attributes']

In [223]:
full_chicken_barn_paf = chicken_barn_paf_filt.merge(chicken_data_cds, on='product_accession', how='inner')

For Sankey diagrams, we are interested in the coordinates.

In [ ]:
sankey_chicken = full_chicken_barn_paf[['qname', 'chromosome', 'genomic_accession', 'tname']]


sankey_chicken['chr'] = np.where(
    sankey_chicken['chromosome'].isna(),
    'unplaced',
    sankey_chicken['chromosome']
)

sankey_chicken['chromosome'] = np.where(
    sankey_chicken['chromosome'].isna(),
    'unplaced_' + sankey_chicken['genomic_accession'].astype(str),
    sankey_chicken['chromosome']
)

In [ ]:
sankey_chicken.to_csv('pafs/sankey_chicken_to_barnowlFA.tsv', sep='\t', index=False)

## Zebra finch - mother assembly

In [ ]:
# load zebra finch annotation 
zfinch_data = pd.read_csv('refdata/GCF_048771995.1_bTaeGut7.mat_feature_table.txt', sep='\t', header=0)
zfinch_data_cds = zfinch_data[zfinch_data['# feature'] == 'CDS']

# load zebra finch to barn owl mother PAF & format the columns
zfinch_barnMO = pd.read_csv('pafs/barnowlMO_zfinch.paf', sep='\t', header=None)
zfinch_barnMO_paf = format_paf(zfinch_barnMO)

In [ ]:
# percent identity 
zfinch_barnMO_paf['p_id'] = (zfinch_barnMO_paf['nmatch']/zfinch_barnMO_paf['alen']) * 100
zfinch_barnMO_paf['p_cov'] = ((zfinch_barnMO_paf['qend'] - zfinch_barnMO_paf['qstart'])/zfinch_barnMO_paf['qlen']) * 100

In [309]:
zfinch_barnMO_paf_filt = zfinch_barnMO_paf[(zfinch_barnMO_paf['p_cov'] > 90) & (zfinch_barnMO_paf['p_id'] > 70)]
print('raw alignments:', zfinch_barnMO_paf.shape[0])
print('alignments after filtering:', zfinch_barnMO_paf_filt.shape[0])

raw alignments: 49244
alignments after filtering: 43408


### Merge paf data with feature information

In [ ]:
# change column names to avoid duplicates & allow merging
zfinch_barnMO_paf_filt['product_accession'] = zfinch_barnMO_paf_filt['qname']

zfinch_data_cds.columns = ['feature', 'class', 'assembly', 'assembly_unit', 'seq_type',
       'chromosome', 'genomic_accession', 'feature_start', 'feature_end', 'feature_strand',
       'product_accession', 'non-redundant_refseq', 'related_accession',
       'name', 'symbol', 'GeneID', 'locus_tag', 'feature_interval_length',
       'product_length', 'attributes']

In [452]:
full_zfinch_barnMO_paf = zfinch_barnMO_paf_filt.merge(zfinch_data_cds, on='product_accession', how='inner')

Prepare Sankey diagrams

In [ ]:
sankey_zfinchMO = full_zfinch_barnMO_paf[['qname', 'chromosome', 'genomic_accession', 'tname']]

In [ ]:
sankey_zfinchMO.to_csv('pafs/sankey_zfinch_to_barnowlMO.tsv', sep='\t', index=False)

## Zebra finch - father assembly

In [ ]:
# load zebra finch to barn owl father PAF & format the columns
zfinch_barn = pd.read_csv('pafs/barnowl_zfinch.paf', sep='\t', header=None)
zfinch_barn_paf = format_paf(zfinch_barn)

In [ ]:
# percent identity 
zfinch_barn_paf['p_id'] = (zfinch_barn_paf['nmatch']/zfinch_barn_paf['alen']) * 100
# % of reference protein covered
zfinch_barn_paf['p_cov'] = ((zfinch_barn_paf['qend'] - zfinch_barn_paf['qstart'])/zfinch_barn_paf['qlen']) * 100

In [337]:
zfinch_barn_paf_filt = zfinch_barn_paf[(zfinch_barn_paf['p_cov'] > 90) & (zfinch_barn_paf['p_id'] > 70)]
print('raw alignments:', zfinch_barn_paf.shape[0])
print('alignments after filtering:', zfinch_barn_paf_filt.shape[0])

raw alignments: 49107
alignments after filtering: 43093


### Merge paf data with feature information

In [ ]:
# change column names to avoid duplicates & allow merging
zfinch_barn_paf_filt['product_accession'] = zfinch_barn_paf_filt['qname']
zfinch_data_cds.columns = ['feature', 'class', 'assembly', 'assembly_unit', 'seq_type',
       'chromosome', 'genomic_accession', 'feature_start', 'feature_end', 'feature_strand',
       'product_accession', 'non-redundant_refseq', 'related_accession',
       'name', 'symbol', 'GeneID', 'locus_tag', 'feature_interval_length',
       'product_length', 'attributes']

In [343]:
full_zfinch_barn_paf = zfinch_barn_paf_filt.merge(zfinch_data_cds, on='product_accession', how='inner')

For Sankey diagrams, we are interested in the coordinates. No unplaced scaffolds in this annotation. 

In [ ]:
sankey_zfinch = full_zfinch_barn_paf[['qname', 'chromosome', 'genomic_accession', 'tname']]

In [ ]:
sankey_zfinch.to_csv('C:pafs/sankey_zfinch_to_barnowlFA.tsv', sep='\t', index=False)